<a href="https://colab.research.google.com/github/edilsonchaves/MVP-Machine-Learning-Analytics/blob/main/OscarResearchML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Descrição do Tabalho**

Neste trabalho busca fazer um levantamento sobre a maior premiação do oscar desde a sua primeira edição que ocorreu no ano de 1929 referente a filmes lançados nos anos de 1927/1928 até a edição de numero 97 que ocorreu no ano de 2026. Este trabalho é uma continuação do executado na sprint de analise de dados e boas práticas.
<br><br>

# **Coleta de dados**

Os dados foram coletados na plataforma kaggle no link : https://www.kaggle.com/datasets/unanimad/the-oscar-award e os dados brutos foram armazenados neste repositório git




# Imports

In [464]:
import requests
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Dados Brutos

In [465]:
url = "https://raw.githubusercontent.com/edilsonchaves/MVP-Machine-Learning-Analytics/refs/heads/main/full_data_2026.csv"
df = pd.read_csv(url, sep="\t")
display(df)

,Ceremony,Year,Class,CanonicalCategory,Category,Film,FilmId,Name,Nominees,NomineeIds,Winner,Detail,Note,Citation
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Noose|The Patent Leather Kid,tt0019217|tt0018253,Richard Barthelmess,Richard Barthelmess,nm0001932,NaN,Nickie Elkins|The Patent Leather Kid,NaN,NaN
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,ACTOR,The Last Command|The Way of All Flesh,tt0019071|tt0019553,Emil Jannings,Emil Jannings,nm0417837,True,General Dolgorucki [Grand Duke Sergius Alexand...,NaN,NaN
2,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,A Ship Comes In,tt0018389,Louise Dresser,Louise Dresser,nm0237571,NaN,Mrs. Pleznik,NaN,NaN
3,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,7th Heaven|Street Angel|Sunrise,tt0018379|tt0019429|tt0018455,Janet Gaynor,Janet Gaynor,nm0310980,True,Diane|Angela|The Wife,NaN,NaN
4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,ACTRESS,Sadie Thompson,tt0019344,Gloria Swanson,Gloria Swanson,nm0841797,NaN,Sadie Thompson,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12132,98,2025,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,NaN,NaN,Benjamin Graf,NaN,True,NaN,dxRevive Pro has transformed modern dialog res...,"To BENJAMIN GRAF for the design, engineering a..."
12133,98,2025,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,NaN,NaN,John Ellwood|Jeff Bloom,NaN,True,NaN,Titan pioneered the auto-assembly of digital a...,To JOHN ELLWOOD for the innovative rules and h...
12134,98,2025,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,NaN,NaN,Marc Specter,NaN,True,NaN,With an intuitive user interface and transcrip...,To MARC SPECTER for the design and development...
12135,98,2025,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,NaN,NaN,Justin Webster,NaN,True,NaN,Providing detailed insight into differences be...,To JUSTIN WEBSTER for the design and engineeri...


# **Pré Processamento**

Antes de inicializar as análises dos dados algumas modificação foram necessárias de serem realizadas para melhorar a qualidade dos dados

## Pré processamento Data Frame
A primeira etapa de pré processamento dos dados foi avaliar quais as colunas existentes na tabela para posteriormente poder escolher aquelas que mais poderiam ajudar nas respostas das questões. Por isso que das dezesseis colunas existentes apenas sete foram escolhidas sendo elas as

* Ceremony: Numero da cerimônia ao oscar
* Year: Ano de lançamento do filme que está sendo homenageado
* Canonical Category: Categoria da premiação
* Film: Nome do filme
* Winner: Se a indicação terminou em vitória ou não
<br>
Além destas colunas foi necessário acrescentar mais 1 coluna

* Genre: Genero do Filme que foi Indicado ao oscar

Na coluna de vencedores os valores 0 indica que a indicação não conquistou o prêmio e 1 é a indicação vencedora

In [466]:
df.insert(8, "Genre", "Genre of Film")

In [467]:
df.loc[df["Name"] == "Nugent Slaughter", "Film"] = "The Jazz Singer"
df.loc[df["Name"] == "Ralph Hammeras", "Film"] = "Undefined Film"

In [468]:
df_Selected_Colluns = ["Ceremony", "Year", "Class", "CanonicalCategory", "Film", "Genre", "Winner"]
df_With_Colluns = df[df_Selected_Colluns].copy()

In [469]:
df_With_Colluns["Winner"] = df_With_Colluns["Winner"].fillna(0)
df_With_Colluns["Winner"] = df_With_Colluns["Winner"].replace(True, 1)

/tmp/ipykernel_2618/1339843477.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_With_Colluns["Winner"] = df_With_Colluns["Winner"].replace(True, 1)


In [470]:
display(df_With_Colluns)

,Ceremony,Year,Class,CanonicalCategory,Film,Genre,Winner
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,The Noose|The Patent Leather Kid,Genre of Film,0
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,The Last Command|The Way of All Flesh,Genre of Film,1
2,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,A Ship Comes In,Genre of Film,0
3,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,7th Heaven|Street Angel|Sunrise,Genre of Film,1
4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,Sadie Thompson,Genre of Film,0
...,...,...,...,...,...,...,...
12132,98,2025,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,Genre of Film,1
12133,98,2025,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,Genre of Film,1
12134,98,2025,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,Genre of Film,1
12135,98,2025,SciTech,SCIENTIFIC AND TECHNICAL AWARD (Technical Achi...,NaN,Genre of Film,1


## **Pré Processamento dos Dados**

Posteriormente que foi trabalhado consistiu em avaliar quais as Categorias de premiação do Oscar seriam trabalhadas para isso foram escolhidas 2 categorias no contexto de filmes sendo estas

* Melhor Filme
* Melhor Filme Internacional

Além de quatro no contexto de premiações para pessoas que foram

* Melhor Ator Principal tanto para Ator como Atriz
* Melhor Ator Coadjuvante tanto para Ator como Atriz
* Melhor Direção
* Melhor Roteiro Adaptado ou Original.

In [471]:
df_Selected_Categorys = ["BEST PICTURE",
                          "INTERNATIONAL FEATURE FILM",
                          "DIRECTING",
                          "ACTOR IN A LEADING ROLE",
                          "ACTRESS IN A LEADING ROLE",
                          "ACTOR IN A SUPPORTING ROLE",
                          "ACTRESS IN A SUPPORTING ROLE",
                          "WRITING (Adapted Screenplay)",
                          "WRITING (Original Story)",
                          #"MUSIC (Original Song)",
                          #"MUSIC (Original Score)",
                          #"VISUAL EFFECTS",
                          #"ART DIRECTION",
                          #"SOUND MIXING",
                          #"COSTUME DESIGN"
                          ]
df_Category_Research = df_With_Colluns[df_With_Colluns["CanonicalCategory"].isin(df_Selected_Categorys)]

## Resultado
Tais pre processamentos reduziu a amostragem para 6186 entradas e 7 colunas como mostrado no diplay abaixo

In [472]:
df_Official = df_Category_Research.copy()
display(df_Official)

,Ceremony,Year,Class,CanonicalCategory,Film,Genre,Winner
0,1,1927/28,Acting,ACTOR IN A LEADING ROLE,The Noose|The Patent Leather Kid,Genre of Film,0
1,1,1927/28,Acting,ACTOR IN A LEADING ROLE,The Last Command|The Way of All Flesh,Genre of Film,1
2,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,A Ship Comes In,Genre of Film,0
3,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,7th Heaven|Street Angel|Sunrise,Genre of Film,1
4,1,1927/28,Acting,ACTRESS IN A LEADING ROLE,Sadie Thompson,Genre of Film,0
...,...,...,...,...,...,...,...
12108,98,2025,Writing,WRITING (Adapted Screenplay),Bugonia,Genre of Film,0
12109,98,2025,Writing,WRITING (Adapted Screenplay),Frankenstein,Genre of Film,0
12110,98,2025,Writing,WRITING (Adapted Screenplay),Hamnet,Genre of Film,0
12111,98,2025,Writing,WRITING (Adapted Screenplay),One Battle after Another,Genre of Film,1


In [473]:
df_Official["Class"].unique()

array(['Acting', 'Title', 'Writing', 'Directing'], dtype=object)

In [474]:
df_Official.groupby("Class")["CanonicalCategory"].unique()

,CanonicalCategory
Class,
Acting,"[ACTOR IN A LEADING ROLE, ACTRESS IN A LEADING..."
Directing,[DIRECTING]
Title,"[BEST PICTURE, INTERNATIONAL FEATURE FILM]"
Writing,"[WRITING (Adapted Screenplay), WRITING (Origin..."


In [475]:
df_best_pictures_ML= df_Official.copy()
df_best_pictures_ML = (
    df_Official[df_Official["CanonicalCategory"] == "BEST PICTURE"]
    [["Film", "Year"]]
    .reset_index(drop=True)
)

#df_best_pictures_ML = pd.get_dummies(
#    df_best_pictures_ML,
#    columns=["Genre"],
#    prefix="Genre",
#    drop_first=True,
#    dtype=int
#)


for category in df_Selected_Categorys:
    df_best_pictures_ML[category] = 0

for category in df_Selected_Categorys:

    nominated_films = set(
        df_Official[
            df_Official["CanonicalCategory"] == category
        ]["Film"].dropna()
    )

    df_best_pictures_ML.loc[
        df_best_pictures_ML["Film"].isin(nominated_films),
        category
    ] = 1

total_nominations = (
    df_Official
    .groupby("Film")
    .size()
    .reset_index(name="Total_Nominations")
)

df_best_pictures_ML = df_best_pictures_ML.merge(
    total_nominations,
    on="Film",
    how="left"
)

winners = (
    df_Official[df_Official["CanonicalCategory"] == "BEST PICTURE"]
    [["Film", "Year", "Winner"]]
    .drop_duplicates()
)

df_best_pictures_ML = df_best_pictures_ML.merge(
    winners,
    on=["Film", "Year"],
    how="left"
)

display(df_best_pictures_ML)

,Film,Year,BEST PICTURE,INTERNATIONAL FEATURE FILM,DIRECTING,ACTOR IN A LEADING ROLE,ACTRESS IN A LEADING ROLE,ACTOR IN A SUPPORTING ROLE,ACTRESS IN A SUPPORTING ROLE,WRITING (Adapted Screenplay),WRITING (Original Story),Total_Nominations,Winner
0,The Racket,1927/28,1,0,0,0,0,0,0,0,0,1,0
1,7th Heaven,1927/28,1,0,0,0,0,0,0,1,0,2,0
2,Wings,1927/28,1,0,0,0,0,0,0,0,0,1,1
3,Alibi,1928/29,1,0,0,1,0,0,0,0,0,2,0
4,In Old Arizona,1928/29,1,0,1,1,0,0,0,0,0,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
616,One Battle after Another,2025,1,0,1,1,0,1,1,1,0,7,1
617,The Secret Agent,2025,1,1,0,1,0,0,0,0,0,3,0
618,Sentimental Value,2025,1,1,1,0,1,1,1,0,0,7,0
619,Sinners,2025,1,0,1,1,0,1,1,0,0,5,0


In [476]:
df_best_pictures_ML["Year"] = (
    df_best_pictures_ML["Year"]
    .astype(str)
    .str[:4]
    .astype(int)
)

In [477]:
train_test_split_year = 1996
train_df = df_best_pictures_ML[df_best_pictures_ML["Year"] <= train_test_split_year]
test_df = df_best_pictures_ML[df_best_pictures_ML["Year"] > train_test_split_year]

In [478]:
x_train = train_df.drop(columns=["Film", "Winner", "Year"])
y_train = train_df["Winner"]

x_test = test_df.drop(columns=["Film", "Winner", "Year"])
y_test = test_df["Winner"]

In [479]:
def evaluate_model(modelName, model, X_train, X_test, Y_train, Y_test):
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  y_prob = model.predict_proba(X_test)[:, 1]

  acc = accuracy_score(Y_test, y_pred)
  f1 = f1_score(Y_test, y_pred)
  precision = precision_score(Y_test, y_pred)
  recall = recall_score(Y_test, y_pred)
  roc_auc = roc_auc_score(Y_test, y_pred)
  cm = confusion_matrix(Y_test, y_pred)
  tn, fp, fn, tp = cm.ravel()

  results = {
    "Model" : modelName,
    "Accuracy" : acc,
    "F1" : f1,
    "Precision" : precision,
    "Recall" : recall,
    "ROC AUC" : roc_auc,
    "True Negative" : tn,
    "False Positive" : fp,
    "False Negative" : fn,
    "True Positive" : tp
  }
  return results


In [480]:
trainning_basics_model = [
  {"name" : "Decision Tree Classifier", "Modelo" : DecisionTreeClassifier(random_state=42)},
  {"name" : "Random Forest Classifier", "Modelo" : RandomForestClassifier(random_state=42)},
  {"name" : "Logistic Regression", "Modelo" : LogisticRegression(random_state=42, max_iter=1000)},
  {"name" : "K Neighbors Classifier", "Modelo" : KNeighborsClassifier(n_neighbors=5)},
  {"name" : "SVC", "Modelo" : SVC(probability=True, random_state=42)}
]

all_basics_results = []
for model in trainning_basics_model:
  result = evaluate_model(model["name"], model["Modelo"], x_train, x_test, y_train, y_test)
  all_basics_results.append(result)

df_basics_result = pd.DataFrame(all_basics_results)
display(df_basics_result)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,Model,Accuracy,F1,Precision,Recall,ROC AUC,True Negative,False Positive,False Negative,True Positive
0,Decision Tree Classifier,0.847222,0.153846,0.300000,0.103448,0.533008,180,7,26,3
1,Random Forest Classifier,0.847222,0.195122,0.333333,0.137931,0.547575,179,8,25,4
2,Logistic Regression,0.861111,0.000000,0.000000,0.000000,0.497326,186,1,29,0
3,K Neighbors Classifier,0.847222,0.195122,0.333333,0.137931,0.547575,179,8,25,4
4,SVC,0.865741,0.000000,0.000000,0.000000,0.500000,187,0,29,0


In [481]:
trainning_modify_model = [
  {"name" : "Decision Tree Classifier", "Modelo" : DecisionTreeClassifier(random_state=42, class_weight="balanced")},
  {"name" : "Random Forest Classifier", "Modelo" : RandomForestClassifier(random_state=42, class_weight="balanced")},
  {"name" : "Logistic Regression", "Modelo" : LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced")},
  {"name" : "K Neighbors Classifier", "Modelo" : KNeighborsClassifier(n_neighbors=1)},
  {"name" : "SVC", "Modelo" : SVC(probability=True, random_state=42, class_weight="balanced")}
]

all_modify_results = []
for model in trainning_modify_model:
  result = evaluate_model(model["name"], model["Modelo"], x_train, x_test, y_train, y_test)
  all_modify_results.append(result)

df_modify_results = pd.DataFrame(all_modify_results)
display(df_modify_results)

,Model,Accuracy,F1,Precision,Recall,ROC AUC,True Negative,False Positive,False Negative,True Positive
0,Decision Tree Classifier,0.717593,0.298851,0.224138,0.448276,0.603817,142,45,16,13
1,Random Forest Classifier,0.745370,0.225352,0.190476,0.275862,0.547022,153,34,21,8
2,Logistic Regression,0.620370,0.327869,0.215054,0.689655,0.649640,114,73,9,20
3,K Neighbors Classifier,0.800926,0.218182,0.230769,0.206897,0.549972,167,20,23,6
4,SVC,0.601852,0.232143,0.156627,0.448276,0.536972,117,70,16,13


In [482]:
all_modify_results = []
for p in ["l1"]:
  method_name = "Regression Advanced", p
  modelAdvanced = LogisticRegression(penalty = p,solver="liblinear", random_state=42, max_iter=1000, class_weight="balanced")
  result = evaluate_model(method_name, modelAdvanced, x_train, x_test, y_train, y_test)
  all_modify_results.append(result)
df_modify_results = pd.DataFrame(all_modify_results)
display(df_modify_results)

,Model,Accuracy,F1,Precision,Recall,ROC AUC,True Negative,False Positive,False Negative,True Positive
0,"(Regression Advanced, l1)",0.601852,0.338462,0.217822,0.758621,0.66808,108,79,7,22


In [483]:
all_modify_results = []
for solver in ["liblinear", "lbfgs", "saga"]:
  method_name = "Regression Advanced", solver
  modelAdvanced = LogisticRegression(solver = solver, random_state=42, max_iter=1000, class_weight="balanced")
  result = evaluate_model(method_name, modelAdvanced, x_train, x_test, y_train, y_test)
  all_modify_results.append(result)
df_modify_results = pd.DataFrame(all_modify_results)
display(df_modify_results)

,Model,Accuracy,F1,Precision,Recall,ROC AUC,True Negative,False Positive,False Negative,True Positive
0,"(Regression Advanced, liblinear)",0.606481,0.330709,0.214286,0.724138,0.656187,110,77,8,21
1,"(Regression Advanced, lbfgs)",0.620370,0.327869,0.215054,0.689655,0.649640,114,73,9,20
2,"(Regression Advanced, saga)",0.620370,0.327869,0.215054,0.689655,0.649640,114,73,9,20


In [484]:
all_modify_results = []
for c in [0.1, 1, 5, 10, 20, 50]:
  method_name = "Regression Advanced", c
  modelAdvanced = LogisticRegression(C = c, random_state=42, max_iter=1000, class_weight="balanced")
  result = evaluate_model(method_name, modelAdvanced, x_train, x_test, y_train, y_test)
  all_modify_results.append(result)
df_modify_results = pd.DataFrame(all_modify_results)
display(df_modify_results)

,Model,Accuracy,F1,Precision,Recall,ROC AUC,True Negative,False Positive,False Negative,True Positive
0,"(Regression Advanced, 0.1)",0.652778,0.285714,0.197368,0.517241,0.595519,126,61,14,15
1,"(Regression Advanced, 1)",0.620370,0.327869,0.215054,0.689655,0.649640,114,73,9,20
2,"(Regression Advanced, 5)",0.638889,0.338983,0.224719,0.689655,0.660336,118,69,9,20
3,"(Regression Advanced, 10)",0.638889,0.338983,0.224719,0.689655,0.660336,118,69,9,20
4,"(Regression Advanced, 20)",0.638889,0.338983,0.224719,0.689655,0.660336,118,69,9,20
5,"(Regression Advanced, 50)",0.638889,0.338983,0.224719,0.689655,0.660336,118,69,9,20


In [485]:
all_modify_results = []
for solver in ["liblinear", "lbfgs", "saga"]:
  for c in [0.1, 1, 5, 10, 20, 50]:
    method_name = "Regression Advanced", c, solver
    modelAdvanced = LogisticRegression(solver = solver, C = c, random_state=42, max_iter=1000, class_weight="balanced")
    result = evaluate_model(method_name, modelAdvanced, x_train, x_test, y_train, y_test)
    all_modify_results.append(result)

df_modify_results = pd.DataFrame(all_modify_results)
display(df_modify_results)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,Model,Accuracy,F1,Precision,Recall,ROC AUC,True Negative,False Positive,False Negative,True Positive
0,"(Regression Advanced, 0.1, liblinear)",0.601852,0.328125,0.212121,0.724138,0.653513,109,78,8,21
1,"(Regression Advanced, 1, liblinear)",0.606481,0.330709,0.214286,0.724138,0.656187,110,77,8,21
2,"(Regression Advanced, 5, liblinear)",0.638889,0.338983,0.224719,0.689655,0.660336,118,69,9,20
3,"(Regression Advanced, 10, liblinear)",0.638889,0.338983,0.224719,0.689655,0.660336,118,69,9,20
4,"(Regression Advanced, 20, liblinear)",0.638889,0.338983,0.224719,0.689655,0.660336,118,69,9,20
5,"(Regression Advanced, 50, liblinear)",0.638889,0.338983,0.224719,0.689655,0.660336,118,69,9,20
6,"(Regression Advanced, 0.1, lbfgs)",0.652778,0.285714,0.197368,0.517241,0.595519,126,61,14,15
7,"(Regression Advanced, 1, lbfgs)",0.620370,0.327869,0.215054,0.689655,0.649640,114,73,9,20
8,"(Regression Advanced, 5, lbfgs)",0.638889,0.338983,0.224719,0.689655,0.660336,118,69,9,20
9,"(Regression Advanced, 10, lbfgs)",0.638889,0.338983,0.224719,0.689655,0.660336,118,69,9,20


In [486]:
from sklearn.preprocessing import StandardScaler

def importance_variable_regression_model(modelName, model, X_train, X_test, Y_train, Y_test):
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  y_prob = model.predict_proba(X_test)[:, 1]
  importance = pd.DataFrame({
      "Feature": X_train.columns,
      "Coefficient": model.coef_[0]
  })

  display(importance)


def importance_variable_random_forest_model(modelName, model, X_train, X_test, Y_train, Y_test):
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  y_prob = model.predict_proba(X_test)[:, 1]
  importance = pd.DataFrame({
      "Feature": X_train.columns,
      "Importance": model.feature_importances_
  })

  display(importance)

In [487]:
importance_variable_regression_model("Regression", LogisticRegression(random_state=42, max_iter=1000, class_weight="balanced"), x_train, x_test, y_train, y_test)

,Feature,Coefficient
0,BEST PICTURE,0.001242
1,INTERNATIONAL FEATURE FILM,-0.288585
2,DIRECTING,1.889497
3,ACTOR IN A LEADING ROLE,0.313549
4,ACTRESS IN A LEADING ROLE,-0.397199
5,ACTOR IN A SUPPORTING ROLE,0.045047
6,ACTRESS IN A SUPPORTING ROLE,-0.782964
7,WRITING (Adapted Screenplay),0.001739
8,WRITING (Original Story),-0.383677
9,Total_Nominations,0.383459


In [488]:
importance_variable_random_forest_model("Random Forest", RandomForestClassifier(random_state=42, class_weight="balanced"), x_train, x_test, y_train, y_test)

,Feature,Importance
0,BEST PICTURE,0.000000
1,INTERNATIONAL FEATURE FILM,0.008208
2,DIRECTING,0.235870
3,ACTOR IN A LEADING ROLE,0.093206
4,ACTRESS IN A LEADING ROLE,0.070334
5,ACTOR IN A SUPPORTING ROLE,0.081634
6,ACTRESS IN A SUPPORTING ROLE,0.071122
7,WRITING (Adapted Screenplay),0.080841
8,WRITING (Original Story),0.031080
9,Total_Nominations,0.327704
